# Demo - Sampling Knobs in Text Generation
**Day 3 - Session 2**

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/kpassoubady/gen-ai-companion-intuite/blob/main/day3/demos/notebooks/demo-sampling-knobs.ipynb)

**Goal:** Show how temperature, top-k, top-p, and beam search change generated
text.

We load a small GPT-2 model and generate from one prompt under several decoding
settings, printing each result so the effect of every knob is visible. Greedy
and low temperature stay repetitive and safe; higher temperature and sampling
add variety; beam search favors fluent, high-probability text.

No API key needed. GPT-2 downloads once (~500 MB), then caches.

In [1]:
# Scoped install for this demo (safe to re-run; often preinstalled on Colab).
%pip install -q torch transformers

In [2]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, set_seed

MODEL_NAME = "gpt2"
PROMPT = "The best way to learn programming is"
NEW_TOKENS = 30

## 1. Load GPT-2 and a generate helper
`generate` runs one continuation of the prompt under whatever decoding kwargs
we pass, re-seeding first so each run is reproducible.

In [3]:
def generate(model, tokenizer, label, **kwargs):
    """Generate one continuation of the prompt and print it under a label."""
    set_seed(42)
    inputs = tokenizer(PROMPT, return_tensors="pt")
    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=NEW_TOKENS,
            pad_token_id=tokenizer.eos_token_id,
            **kwargs,
        )
    text = tokenizer.decode(output_ids[0], skip_special_tokens=True)
    print(f"\n[{label}]")
    print(f"  {text}")

In [4]:
print(f"Loading {MODEL_NAME} (downloads once, then cached)...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForCausalLM.from_pretrained(MODEL_NAME)
model.eval()

print(f"\nPrompt: {PROMPT!r}")

Loading gpt2 (downloads once, then cached)...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:138: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
  warnings.warn(f"\nError while fetching `HF_TOKEN` secret value from your vault: '{str(e)}'.")


config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.04M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

... (2 outputs left out)

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]


Prompt: 'The best way to learn programming is'


## 2. Greedy and temperature
Greedy always picks the top token (repetitive). Low temperature stays focused;
high temperature adds variety and risk.

In [7]:
generate(model, tokenizer, "greedy (do_sample=False)", do_sample=False)
generate(model, tokenizer, "temperature=0.2 (focused)",
         do_sample=True, temperature=0.3)
generate(model, tokenizer, "temperature=1.2 (wild)",
         do_sample=True, temperature=2.0)


[greedy (do_sample=False)]
  The best way to learn programming is to learn how to use the language.

The best way to learn programming is to learn how to use the language. The best way to learn

[temperature=0.2 (focused)]
  The best way to learn programming is to learn it from someone who has never taught you anything, and who has never taught you anything new.

The best way to learn programming is

[temperature=1.2 (wild)]
  The best way to learn programming is with Java," a teacher at MWC will attest . And to see what's new comes via my research. Today's posts give examples that come full


## 3. Top-k, top-p, and beam search
Top-k and top-p (nucleus) restrict sampling to likely tokens. Beam search
explores several paths and favors fluent, high-probability text.

In [8]:
generate(model, tokenizer, "top_k=10", do_sample=True, top_k=10)
generate(model, tokenizer, "top_p=0.90 (nucleus)",
         do_sample=True, top_p=0.90)
generate(model, tokenizer, "beam search (num_beams=5)",
         do_sample=False, num_beams=5, early_stopping=True)


[top_k=10]
  The best way to learn programming is to go into a programming school in your area, or to go into one of the local libraries and learn from them.

I've seen many

[top_p=0.90 (nucleus)]
  The best way to learn programming is to just ask questions, and you'll learn something about what to do and how to do it. For example, here is the most important question for

[beam search (num_beams=5)]
  The best way to learn programming is to learn how to code.

The best way to learn programming is to learn how to code. The best way to learn programming is to learn


## Expected output
- Greedy and low temperature produce safe, sometimes repetitive text.
- High temperature and sampling add variety (and occasional nonsense).
- Beam search reads as the most fluent.

**Takeaway:** same prompt, different knobs, different text. That is decoding.